[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/poc1_endosymbiosis.ipynb)

# POC 1: Organelle-Specialist Fusion (Endosymbiosis)

**Hypothesis**: Single-organelle specialists capture orthogonal features
(conv = local n-grams, attention = long-range deps, monarch = sub-quadratic global mixing).
Fusing them into a gated multi-organelle model produces a better model than training
from scratch at the same parameter budget.

**Setup**:
- 3 specialists (~200-290K each): ConvSpec, AttnSpec, MonarchSpec
- Fused target (~850K): 3-organelle SymbioGPT with transplanted weights
- Teacher: JuliaSLM 5M (val_loss=3.54)
- Data: 266M token curated philosophy corpus, BPE-2000, ctx=256

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup — install dependencies
!pip install -q wandb huggingface_hub

In [ ]:
# 2. GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 3. W&B + HF login (MANDATORY per CLAUDE.md)
import os
import wandb
from huggingface_hub import login as hf_login

# Prefer Colab secrets, fall back to env vars, then interactive prompt
try:
    from google.colab import userdata
    os.environ.setdefault("WANDB_API_KEY", userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except (ImportError, Exception):
    pass  # Not in Colab or secrets not configured

wandb.login()
hf_login(token=os.environ.get("HF_TOKEN"), add_to_git_credential=False)

In [ ]:
# 4. Download data, model code, and JuliaSLM teacher weights from HuggingFace
import os, sys, math, time, copy, json
import numpy as np
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Tuple
from huggingface_hub import hf_hub_download, HfApi, create_repo
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_REPO = 'LisaMegaWatts/SymbioGPT-10M'
SLM_REPO = 'LisaMegaWatts/JuliaSLM'
os.makedirs('data', exist_ok=True)

print('Downloading symbio_model.py...')
hf_hub_download(repo_id=DATA_REPO, filename='symbio_model.py', local_dir='.')

print('Downloading pre-tokenized data...')
hf_hub_download(repo_id=DATA_REPO, filename='data/train_curated.txt.tokens.pt', local_dir='.')
hf_hub_download(repo_id=DATA_REPO, filename='data/val.txt.tokens.pt', local_dir='.')

print('Downloading JuliaSLM weights (NPZ)...')
hf_hub_download(repo_id=SLM_REPO, filename='juliaslm_weights.npz', local_dir='.')

CTX = 256
print('Loading tokens...')
train_tokens = torch.load('data/train_curated.txt.tokens.pt', weights_only=True).tolist()
val_tokens = torch.load('data/val.txt.tokens.pt', weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f'Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)')
print(f'Val: {len(val_inputs):,} seqs')
del train_tokens, val_tokens

In [ ]:
# 5. Model definitions + JuliaSLM teacher + evaluation
#
# SymbioGPT is imported from symbio_model.py (downloaded from HF in cell 4).
# JuliaSLM teacher is defined inline (uses Julia-compatible attention reshape).

import sys
sys.path.insert(0, '.')

from symbio_model import (
    SymbioConfig, SymbioGPT, compute_symbio_params, compute_gate_entropy,
    RMSNorm as SymbioRMSNorm, SwiGLU, CausalDepthwiseConv1d, MonarchMatrix,
    LongConv, CausalSelfAttention, OrganelleGate, SkipGate,
    SymbioSequenceMixer, SymbioBlock, RotaryEmbedding,
)

# --- JuliaSLM teacher (Julia column-major reshape) ---

@dataclass
class JuliaSLMConfig:
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 4
    head_dim: int = 64
    ffn_inner: int = 640
    context_length: int = 256
    vocab_size: int = 2000
    weight_tying: bool = True
    rope_base: float = 10000.0


class _RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class _RoPE(nn.Module):
    def __init__(self, dim, max_seq=256, base=10000.0):
        super().__init__()
        freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        angles = torch.outer(torch.arange(max_seq).float(), freqs)
        self.register_buffer('cos_cache', angles.cos())
        self.register_buffer('sin_cache', angles.sin())
    def forward(self, x):
        T = x.size(2)
        half = x.size(-1) // 2
        x1, x2 = x[..., :half], x[..., half:]
        cos = self.cos_cache[:T, :half].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cache[:T, :half].unsqueeze(0).unsqueeze(0)
        return torch.cat([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1)


class _JuliaSLMAttn(nn.Module):
    """MHA with Julia column-major reshape: view(B, H, T, HD)."""
    def __init__(self, d, n_heads, hd):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, hd
        total = n_heads * hd
        self.wq = nn.Linear(d, total, bias=False)
        self.wk = nn.Linear(d, total, bias=False)
        self.wv = nn.Linear(d, total, bias=False)
        self.wo = nn.Linear(total, d, bias=False)
    def forward(self, x, rope, mask):
        B, T, _ = x.shape
        H, HD = self.n_heads, self.head_dim
        q = self.wq(x).view(B, H, T, HD)  # Julia column-major match
        k = self.wk(x).view(B, H, T, HD)
        v = self.wv(x).view(B, H, T, HD)
        q, k = rope(q), rope(k)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(HD)
        attn = F.softmax(attn + mask, dim=-1)
        out = torch.matmul(attn, v).contiguous().view(B, T, H * HD)
        return self.wo(out)


class _JuliaSLMBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = _RMSNorm(cfg.d_model)
        self.attn = _JuliaSLMAttn(cfg.d_model, cfg.n_heads, cfg.head_dim)
        self.ln2 = _RMSNorm(cfg.d_model)
        # SwiGLU with explicit inner dim
        self.w1 = nn.Linear(cfg.d_model, cfg.ffn_inner, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.ffn_inner, bias=False)
        self.w2 = nn.Linear(cfg.ffn_inner, cfg.d_model, bias=False)
    def forward(self, x, rope, mask):
        x = x + self.attn(self.ln1(x), rope, mask)
        h = self.ln2(x)
        x = x + self.w2(F.silu(self.w1(h)) * self.v(h))
        return x


class JuliaSLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.rope = _RoPE(cfg.head_dim, cfg.context_length, cfg.rope_base)
        self.blocks = nn.ModuleList([_JuliaSLMBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = _RMSNorm(cfg.d_model)
    def forward(self, input_ids):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device, dtype=x.dtype), diagonal=1)
        for block in self.blocks:
            x = block(x, self.rope, mask)
        x = self.ln_f(x)
        return F.linear(x, self.tok_emb.weight)


def load_juliaslm_teacher():
    """Load JuliaSLM 5M from NPZ."""
    cfg = JuliaSLMConfig()
    model = JuliaSLM(cfg)
    npz = np.load('juliaslm_weights.npz')
    sd = {}
    model_keys = set(model.state_dict().keys())
    for key in npz.files:
        if key.startswith('_hp_'):
            continue
        t = torch.from_numpy(npz[key].copy())
        if key in model_keys:
            sd[key] = t
        elif key + '.weight' in model_keys:
            sd[key + '.weight'] = t
    # Map FFN keys: NPZ uses blocks.N.ffn.{w1,v,w2}, model uses blocks.N.{w1,v,w2}
    for key in list(npz.files):
        if key.startswith('_hp_'):
            continue
        t = torch.from_numpy(npz[key].copy())
        mapped = key.replace('.ffn.w1', '.w1').replace('.ffn.v', '.v').replace('.ffn.w2', '.w2')
        if mapped + '.weight' in model_keys and mapped + '.weight' not in sd:
            sd[mapped + '.weight'] = t
        elif mapped in model_keys and mapped not in sd:
            sd[mapped] = t
    missing, unexpected = model.load_state_dict(sd, strict=False)
    real_missing = [k for k in missing if 'cache' not in k]
    if real_missing:
        print(f'WARNING: Missing weights: {real_missing}')
    n_params = sum(p.numel() for p in model.parameters())
    print(f'JuliaSLM teacher loaded: {n_params:,} params')
    return model


@torch.no_grad()
def evaluate(model, val_inputs, val_labels, batch_size=64):
    """Compute val loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    dev = next(model.parameters()).device
    for i in range(0, len(val_inputs), batch_size):
        batch_in = val_inputs[i:i+batch_size].to(dev)
        batch_tgt = val_labels[i:i+batch_size].to(dev)
        logits = model(batch_in)
        B, T, V = logits.shape
        loss = F.cross_entropy(
            logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction='sum'
        )
        total_loss += loss.item()
        total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl


# Verify teacher
teacher = load_juliaslm_teacher().to(device)
teacher.eval()
t_loss, t_ppl = evaluate(teacher, val_inputs, val_labels)
print(f'Teacher baseline: val_loss={t_loss:.4f} ppl={t_ppl:.1f} (expected ~3.54/34.5)')

In [ ]:
# 6. Specialist configs + training function
#
# All specialists: d=64, L=3, H=2, hd=32 (shared head_dim for clean transplant)
# Fused target: d=128, L=3, H=4, hd=32 (~849K params)
#
# NOTE: AMP disabled for small Monarch models — 16x16x16 ops are too small
# to saturate tensor cores, and f16 conversion overhead makes it SLOWER.
# (See: SymbioSLM training notes — v2 f16 AMP was 2.45x slower than v3 f32)

SPECIALIST_CONFIGS = {
    'ConvSpec': SymbioConfig(
        d_model=64, n_layers=3, n_heads=2, head_dim=32,
        organelles=('causal_conv',), context_length=256,
        vocab_size=2000, n_monarch_heads=1, ffn_mult=4,
        free_energy_beta=0.0,
    ),
    'AttnSpec': SymbioConfig(
        d_model=64, n_layers=3, n_heads=2, head_dim=32,
        organelles=('attention',), context_length=256,
        vocab_size=2000, n_monarch_heads=1, ffn_mult=4,
        free_energy_beta=0.0,
    ),
    'MonarchSpec': SymbioConfig(
        d_model=64, n_layers=3, n_heads=2, head_dim=32,
        organelles=('monarch',), context_length=256,
        vocab_size=2000, n_monarch_heads=1, ffn_mult=4,
        free_energy_beta=0.0,
    ),
}

FUSED_CONFIG = SymbioConfig(
    d_model=128, n_layers=3, n_heads=4, head_dim=32,
    organelles=('causal_conv', 'monarch', 'attention'),
    context_length=256, vocab_size=2000, n_monarch_heads=1, ffn_mult=4,
    free_energy_beta=0.0,
)

print('Specialist configs:')
for name, cfg in SPECIALIST_CONFIGS.items():
    print(f'  {name}: {compute_symbio_params(cfg):,} params')
print(f'Fused target: {compute_symbio_params(FUSED_CONFIG):,} params')


def train_model(model, name, n_steps, lr, batch_size=32, warmup_frac=0.1,
                eval_every=500, teacher_model=None, kd_alpha=0.5, kd_temp=2.0,
                wandb_tags=None):
    """Train a model with optional KD from teacher. Returns (model, best_loss, history).

    Uses f32 (no AMP) — small Monarch models are slower with AMP.
    """
    n_params = sum(p.numel() for p in model.parameters())
    warmup_steps = max(1, int(n_steps * warmup_frac))

    run = wandb.init(
        project='symbiogenesis',
        name=f'poc1-{name}',
        config={'name': name, 'n_params': n_params, 'n_steps': n_steps,
                'lr': lr, 'batch_size': batch_size, 'kd': teacher_model is not None,
                'kd_alpha': kd_alpha if teacher_model else None,
                'precision': 'f32'},
        tags=wandb_tags or ['poc1', 'endosymbiosis'],
        reinit='finish_previous',
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1, betas=(0.9, 0.95))
    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(n_steps - warmup_steps, 1)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    n_train = len(train_inputs)
    model.train()
    best_loss = float('inf')
    history = []
    t_start = time.time()
    step = 0

    print(f'\nTraining {name}: {n_params:,} params, {n_steps} steps, lr={lr}')

    while step < n_steps:
        perm = torch.randperm(n_train)
        for i in range(0, n_train, batch_size):
            if step >= n_steps:
                break
            idx = perm[i:i+batch_size]
            batch_in = train_inputs[idx].to(device)
            batch_tgt = train_labels[idx].to(device)

            logits = model(batch_in)
            B, T, V = logits.shape
            ce_loss = F.cross_entropy(logits.reshape(B*T, V), batch_tgt.reshape(B*T))

            if teacher_model is not None:
                with torch.no_grad():
                    t_logits = teacher_model(batch_in)
                kd_loss = F.kl_div(
                    F.log_softmax(logits.reshape(B*T, V) / kd_temp, dim=-1),
                    F.softmax(t_logits.reshape(B*T, V) / kd_temp, dim=-1),
                    reduction='batchmean',
                ) * (kd_temp ** 2)
                loss = kd_alpha * ce_loss + (1 - kd_alpha) * kd_loss
            else:
                loss = ce_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

            if step % 50 == 0:
                wandb.log({'train/loss': loss.item(), 'train/ce': ce_loss.item(),
                           'train/lr': scheduler.get_last_lr()[0]}, step=step)

            if step > 0 and step % eval_every == 0:
                val_loss, val_ppl = evaluate(model, val_inputs, val_labels)
                history.append((step, val_loss, val_ppl))
                wandb.log({'val/loss': val_loss, 'val/ppl': val_ppl}, step=step)
                marker = ' ** BEST **' if val_loss < best_loss else ''
                if val_loss < best_loss:
                    best_loss = val_loss
                    torch.save(model.state_dict(), f'{name}_best.pt')
                elapsed = time.time() - t_start
                print(f'  [{name} step {step:5d}] val={val_loss:.4f} ppl={val_ppl:.1f} ({elapsed:.0f}s){marker}')
                model.train()

            step += 1

    # Final eval
    val_loss, val_ppl = evaluate(model, val_inputs, val_labels)
    history.append((step, val_loss, val_ppl))
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), f'{name}_best.pt')
    elapsed = time.time() - t_start
    print(f'  {name} done ({elapsed:.0f}s): best val_loss={best_loss:.4f}')
    wandb.log({'val/final_loss': best_loss, 'val/final_ppl': val_ppl})
    wandb.finish()

    # Reload best
    best_sd = torch.load(f'{name}_best.pt', map_location=device, weights_only=True)
    model.load_state_dict(best_sd, strict=False)
    model.eval()
    return model, best_loss, history


print('train_model() defined.')

In [ ]:
# 7. Train all 3 specialists (NO KD -- each learns from data only)

SPECIALIST_STEPS = 15_000  # Chinchilla-proportional for ~250K params on 266M tokens
SPECIALIST_LR = 6e-4

specialists = {}  # name -> (model, best_loss, history)

for name, cfg in SPECIALIST_CONFIGS.items():
    model = SymbioGPT(cfg).to(device)
    model, best_loss, history = train_model(
        model, name, n_steps=SPECIALIST_STEPS, lr=SPECIALIST_LR,
        batch_size=32, eval_every=500,
        wandb_tags=['poc1', 'specialist', name.lower()],
    )
    specialists[name] = (model, best_loss, history)
    print(f'{name}: val_loss={best_loss:.4f}\n')

print('\n=== Specialist Results ===')
for name, (_, loss, _) in specialists.items():
    p = compute_symbio_params(SPECIALIST_CONFIGS[name])
    print(f'  {name}: {p:,} params, val_loss={loss:.4f}')

In [ ]:
# 8. Measure specialist orthogonality
#
# If specialists are truly capturing orthogonal features, their output logit
# distributions should differ. We measure pairwise cosine similarity.

print('Measuring specialist orthogonality...')

# Get logits from each specialist on a shared val batch
n_samples = min(256, len(val_inputs))
sample_in = val_inputs[:n_samples].to(device)

spec_logits = {}
for name, (model, _, _) in specialists.items():
    model.eval()
    with torch.no_grad():
        logits = model(sample_in)  # (N, T, V)
        spec_logits[name] = logits.float().reshape(-1, logits.size(-1))  # (N*T, V)

# Pairwise cosine similarity
names = list(spec_logits.keys())
print(f'\nPairwise logit cosine similarity ({n_samples} seqs):')
print(f'{"":>15}', end='')
for n in names:
    print(f'{n:>14}', end='')
print()

for i, n1 in enumerate(names):
    print(f'{n1:>15}', end='')
    for j, n2 in enumerate(names):
        sim = F.cosine_similarity(spec_logits[n1], spec_logits[n2], dim=-1).mean().item()
        print(f'{sim:>14.4f}', end='')
    print()

# Log to W&B
run = wandb.init(project='symbiogenesis', name='poc1-orthogonality',
                 tags=['poc1', 'analysis'], reinit='finish_previous')
for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        if j > i:
            sim = F.cosine_similarity(spec_logits[n1], spec_logits[n2], dim=-1).mean().item()
            wandb.log({f'orthogonality/{n1}_vs_{n2}': sim})
wandb.finish()

del spec_logits  # free memory

In [ ]:
# 9. Organelle transplant function
#
# Transplants organelle-specific weights from single-organelle specialists
# into a multi-organelle target model. This is the core endosymbiosis operation.
#
# Source specialists: d=64, L=3, H=2, hd=32
# Target:            d=128, L=3, H=4, hd=32 (3 organelles)
#
# Projection: top-left submatrix copy + zero-pad (following _project_weights pattern
# from symbiogenesis/fusion.py)

def transplant_organelles(specialists_dict, target_config):
    """Create a multi-organelle model with weights transplanted from specialists.
    
    Args:
        specialists_dict: {name: (model, loss, history)} for ConvSpec, AttnSpec, MonarchSpec
        target_config: SymbioConfig for the fused model
    Returns:
        SymbioGPT with transplanted weights
    """
    conv_model = specialists_dict['ConvSpec'][0]
    attn_model = specialists_dict['AttnSpec'][0]
    monarch_model = specialists_dict['MonarchSpec'][0]
    
    # Find best specialist for shared components (embedding, FFN, norms)
    best_name = min(specialists_dict, key=lambda n: specialists_dict[n][1])
    best_model = specialists_dict[best_name][0]
    print(f'Best specialist for shared components: {best_name} (loss={specialists_dict[best_name][1]:.4f})')
    
    d_src = 64   # specialist d_model
    d_tgt = target_config.d_model  # 128
    h_src = 2    # specialist n_heads
    h_tgt = target_config.n_heads  # 4
    hd = 32      # shared head_dim
    
    # Create fresh target model
    target = SymbioGPT(target_config)
    tgt_sd = target.state_dict()
    
    # Helper: zero-pad a 2D weight matrix
    def pad_2d(src_w, tgt_shape):
        out = torch.zeros(tgt_shape, dtype=src_w.dtype)
        min_r = min(src_w.shape[0], tgt_shape[0])
        min_c = min(src_w.shape[1], tgt_shape[1])
        out[:min_r, :min_c] = src_w[:min_r, :min_c]
        return out
    
    # Helper: zero-pad a 1D weight vector (RMSNorm), fill with val
    def pad_1d(src_w, tgt_len, fill=1.0):
        out = torch.full((tgt_len,), fill, dtype=src_w.dtype)
        min_d = min(src_w.shape[0], tgt_len)
        out[:min_d] = src_w[:min_d]
        return out
    
    # === Embedding ===
    # (vocab, d_src) -> (vocab, d_tgt): copy first d_src cols, fill rest with noise
    src_emb = best_model.state_dict()['tok_emb.weight']
    emb_std = src_emb.std().item()
    new_emb = torch.randn(target_config.vocab_size, d_tgt) * emb_std * 0.02
    new_emb[:, :d_src] = src_emb
    tgt_sd['tok_emb.weight'] = new_emb
    
    # === Per-layer transplant ===
    for layer_i in range(target_config.n_layers):
        # The target SymbioBlock has: seq_mixer.organelle_modules.{causal_conv, monarch, attention}
        # plus seq_mixer.gate, ln1, ln2, ffn, skip1, skip2
        blk = f'blocks.{layer_i}'
        mixer = f'{blk}.seq_mixer'
        
        conv_sd = conv_model.state_dict()
        attn_sd = attn_model.state_dict()
        monarch_sd = monarch_model.state_dict()
        best_sd = best_model.state_dict()
        
        # --- CausalDepthwiseConv1d ---
        # Source: (d_src, 1, kernel_size), Target: (d_tgt, 1, kernel_size)
        conv_key_src = f'{blk}.seq_mixer.organelle_modules.causal_conv.weight'
        conv_key_tgt = f'{mixer}.organelle_modules.causal_conv.weight'
        src_conv_w = conv_sd[conv_key_src]
        tgt_conv_w = torch.zeros_like(tgt_sd[conv_key_tgt])
        tgt_conv_w[:d_src, :, :] = src_conv_w
        tgt_sd[conv_key_tgt] = tgt_conv_w
        
        # --- CausalSelfAttention ---
        # wq/wk/wv: (h_src*hd, d_src) -> (h_tgt*hd, d_tgt)
        # Copy first h_src heads, leave h_tgt-h_src heads at random init
        for proj in ['wq', 'wk', 'wv']:
            src_key = f'{blk}.seq_mixer.organelle_modules.attention.{proj}.weight'
            tgt_key = f'{mixer}.organelle_modules.attention.{proj}.weight'
            src_w = attn_sd[src_key]  # (h_src*hd, d_src)
            tgt_shape = tgt_sd[tgt_key].shape  # (h_tgt*hd, d_tgt)
            tgt_sd[tgt_key] = pad_2d(src_w, tgt_shape)
        
        # wo: (d_src, h_src*hd) -> (d_tgt, h_tgt*hd)
        src_key = f'{blk}.seq_mixer.organelle_modules.attention.wo.weight'
        tgt_key = f'{mixer}.organelle_modules.attention.wo.weight'
        tgt_sd[tgt_key] = pad_2d(attn_sd[src_key], tgt_sd[tgt_key].shape)
        
        # --- MonarchMatrix ---
        # L1, L2: (p, p, p) where p=16. These don't depend on d_model!
        # Monarch operates on token dim, not channel dim. Direct copy.
        for lparam in ['L1', 'L2']:
            src_key = f'{blk}.seq_mixer.organelle_modules.monarch.0.{lparam}'
            tgt_key = f'{mixer}.organelle_modules.monarch.0.{lparam}'
            tgt_sd[tgt_key] = monarch_sd[src_key].clone()
        
        # --- OrganelleGate ---
        # Initialize to uniform (zeros). 3 organelles, d_tgt channels.
        gate_key = f'{mixer}.gate.logits'
        tgt_sd[gate_key] = torch.zeros_like(tgt_sd[gate_key])
        temp_key = f'{mixer}.gate.temperature'
        tgt_sd[temp_key] = torch.ones_like(tgt_sd[temp_key])
        
        # --- SwiGLU FFN (from best specialist) ---
        for ffn_part in ['w1', 'v', 'w2']:
            src_key = f'{blk}.ffn.{ffn_part}.weight'
            tgt_key = f'{blk}.ffn.{ffn_part}.weight'
            tgt_sd[tgt_key] = pad_2d(best_sd[src_key], tgt_sd[tgt_key].shape)
        
        # --- RMSNorm (from best specialist, pad with 1.0) ---
        for ln in ['ln1', 'ln2']:
            src_key = f'{blk}.{ln}.weight'
            tgt_key = f'{blk}.{ln}.weight'
            tgt_sd[tgt_key] = pad_1d(best_sd[src_key], d_tgt, fill=1.0)
        
        # --- SkipGates (keep at 1.0 init) ---
        # Already initialized to 1.0 by SkipGate.__init__
    
    # === Final norm (from best specialist) ===
    tgt_sd['ln_f.weight'] = pad_1d(best_sd['ln_f.weight'], d_tgt, fill=1.0)
    
    # Load transplanted state dict
    target.load_state_dict(tgt_sd, strict=False)
    
    # Count how many params were transplanted vs random
    total_params = sum(p.numel() for p in target.parameters())
    print(f'\nTransplant complete: {total_params:,} total params')
    print(f'  Embedding: {d_src}/{d_tgt} cols from best specialist + noise')
    print(f'  Conv: {d_src}/{d_tgt} channels from ConvSpec')
    print(f'  Attention: {h_src}/{h_tgt} heads from AttnSpec')
    print(f'  Monarch: L1/L2 fully copied from MonarchSpec (p={target_config.p})')
    print(f'  FFN + Norms: from {best_name} with zero-padding')
    print(f'  Gate: uniform (zeros) -- will specialize during fine-tuning')
    
    return target


print('transplant_organelles() defined.')

In [ ]:
# 10. Execute transplant, verify, and fine-tune with KD

# --- Transplant ---
fused_model = transplant_organelles(specialists, FUSED_CONFIG).to(device)

# Verify: forward pass should not produce NaN/Inf
fused_model.eval()
with torch.no_grad():
    test_out = fused_model(val_inputs[:4].to(device))
    assert not torch.isnan(test_out).any(), 'NaN in fused model output!'
    assert not torch.isinf(test_out).any(), 'Inf in fused model output!'
print('Forward pass: OK (no NaN/Inf)')

# Pre-fine-tune evaluation
pre_loss, pre_ppl = evaluate(fused_model, val_inputs, val_labels)
print(f'Pre-fine-tune: val_loss={pre_loss:.4f} ppl={pre_ppl:.1f}')
print(f'  (random init would be ~7.6, specialists are ~{min(s[1] for s in specialists.values()):.2f})')

# Gate weights before fine-tuning
print('\nGate weights (pre-fine-tune, should be ~uniform):')
org_names = list(FUSED_CONFIG.organelles)
for i, w in enumerate(fused_model.get_gate_weights()):
    means = w.mean(dim=1)
    parts = ' | '.join(f'{org_names[j]}={means[j]:.3f}' for j in range(len(org_names)))
    print(f'  Layer {i}: {parts}')

# --- Fine-tune with KD from JuliaSLM teacher ---
print('\n' + '='*60)
print('Fine-tuning fused model with KD from JuliaSLM teacher')
print('='*60)

fused_model, fused_loss, fused_history = train_model(
    fused_model, 'fused-kd', n_steps=15_000, lr=3e-4,
    batch_size=32, eval_every=500, warmup_frac=0.05,
    teacher_model=teacher, kd_alpha=0.5, kd_temp=2.0,
    wandb_tags=['poc1', 'fused', 'kd'],
)

print(f'\nFused model: val_loss={fused_loss:.4f}')

# Gate weights after fine-tuning
print('\nGate weights (post-fine-tune, should show specialization):')
for i, w in enumerate(fused_model.get_gate_weights()):
    means = w.mean(dim=1)
    parts = ' | '.join(f'{org_names[j]}={means[j]:.3f}' for j in range(len(org_names)))
    print(f'  Layer {i}: {parts}')

gate_entropy = compute_gate_entropy(fused_model)
print(f'Gate entropy: {gate_entropy:.4f} (lower = more specialized)')

In [ ]:
# 11. Baselines — same compute budget, different approaches
#
# Total specialist compute: 3 * 15K steps = 45K steps on ~250K-param models
# Total fine-tune compute: 15K steps on ~850K-param model
# Total compute: ~45K + 15K = 60K steps
#
# Baseline A: From-scratch (same target config, 60K steps, no KD)
# Baseline B: KD-only (same target config, 60K steps, KD from teacher, no specialist fusion)

BASELINE_STEPS = 45_000 + 15_000  # compute-matched

# Baseline A: From scratch
print('Training Baseline A: from-scratch (no KD, no specialists)')
baseline_a = SymbioGPT(FUSED_CONFIG).to(device)
baseline_a, baseline_a_loss, baseline_a_hist = train_model(
    baseline_a, 'baseline-scratch', n_steps=BASELINE_STEPS, lr=6e-4,
    batch_size=32, eval_every=1000,
    wandb_tags=['poc1', 'baseline', 'scratch'],
)

# Baseline B: KD only (no specialist fusion)
print('\nTraining Baseline B: KD-only (teacher KD, no specialists)')
baseline_b = SymbioGPT(FUSED_CONFIG).to(device)
baseline_b, baseline_b_loss, baseline_b_hist = train_model(
    baseline_b, 'baseline-kd', n_steps=BASELINE_STEPS, lr=6e-4,
    batch_size=32, eval_every=1000,
    teacher_model=teacher, kd_alpha=0.5, kd_temp=2.0,
    wandb_tags=['poc1', 'baseline', 'kd'],
)

print(f'\n{"="*60}')
print(f'Baseline A (scratch):  val_loss={baseline_a_loss:.4f}')
print(f'Baseline B (KD-only):  val_loss={baseline_b_loss:.4f}')
print(f'Fused (specialists+KD): val_loss={fused_loss:.4f}')

In [ ]:
# 12. Results comparison + gate analysis + HF upload
#
# Handles case where baselines were stopped early or never run.

print('\n' + '='*75)
print('POC 1 RESULTS: Organelle-Specialist Fusion (Endosymbiosis)')
print('='*75)

fused_params = compute_symbio_params(FUSED_CONFIG)

# Check which baselines completed
has_baseline_a = 'baseline_a_loss' in dir() and baseline_a_loss is not None
has_baseline_b = 'baseline_b_loss' in dir() and baseline_b_loss is not None

print(f'\n{"Model":<35} {"Params":>10} {"Val Loss":>10} {"vs Fused":>10}')
print('-' * 67)
for name, (_, loss, _) in specialists.items():
    p = compute_symbio_params(SPECIALIST_CONFIGS[name])
    delta = loss - fused_loss
    print(f'{name + " (specialist)":<35} {p:>10,} {loss:>10.4f} {delta:>+10.4f}')
print(f'{"Fused (specialists + KD)":<35} {fused_params:>10,} {fused_loss:>10.4f} {"---":>10}')
if has_baseline_a:
    print(f'{"Baseline A (scratch, 60K steps)":<35} {fused_params:>10,} {baseline_a_loss:>10.4f} {baseline_a_loss - fused_loss:>+10.4f}')
else:
    print(f'{"Baseline A (scratch)":<35} {fused_params:>10,} {"stopped":>10} {"N/A":>10}')
if has_baseline_b:
    print(f'{"Baseline B (KD-only, 60K steps)":<35} {fused_params:>10,} {baseline_b_loss:>10.4f} {baseline_b_loss - fused_loss:>+10.4f}')
else:
    print(f'{"Baseline B (KD-only)":<35} {fused_params:>10,} {"not run":>10} {"N/A":>10}')
print(f'{"JuliaSLM teacher":<35} {"5.04M":>10} {t_loss:>10.4f} {t_loss - fused_loss:>+10.4f}')

# Success criteria
best_spec_loss = min(s[1] for s in specialists.values())
print(f'\n--- Success Criteria ---')
spec_delta = best_spec_loss - fused_loss
print(f'Fused < best specialist by >=0.1: {"YES" if spec_delta >= 0.1 else "NO"} '
      f'(delta={spec_delta:+.4f})')
if has_baseline_a:
    print(f'Fused < from-scratch baseline:    {"YES" if fused_loss < baseline_a_loss else "NO"} '
          f'(delta={baseline_a_loss - fused_loss:+.4f})')
else:
    print(f'Fused < from-scratch baseline:    INCONCLUSIVE (baseline stopped early)')
if has_baseline_b:
    print(f'Fused < KD-only baseline:         {"YES" if fused_loss < baseline_b_loss else "NO"} '
          f'(delta={baseline_b_loss - fused_loss:+.4f})')
else:
    print(f'Fused < KD-only baseline:         INCONCLUSIVE (baseline not run)')

# Gate analysis
print(f'\n--- Gate Specialization ---')
gate_ent = compute_gate_entropy(fused_model)
print(f'Gate entropy: {gate_ent:.4f} (uniform=1.099, lower=more specialized)')
org_names = list(FUSED_CONFIG.organelles)
print(f'\n{"Layer":<8}', end='')
for n in org_names:
    print(f'{n:>14}', end='')
print(f'{"Dominant":>12}')

for i, w in enumerate(fused_model.get_gate_weights()):
    means = w.mean(dim=1)
    dominant = org_names[means.argmax().item()]
    print(f'Layer {i:<3}', end='')
    for j in range(len(org_names)):
        print(f'{means[j].item():>14.3f}', end='')
    print(f'{dominant:>12}')

# Scaling law context
print(f'\n--- Scaling Law Context ---')
print(f'{"Model":<35} {"Params":>10} {"Val Loss":>10}')
print('-' * 57)
print(f'{"JuliaFluxGPT-1M":.<35} {"1.01M":>10} {"4.446":>10}')
print(f'{"** POC1 Fused **":<35} {fused_params/1e6:>9.2f}M {fused_loss:>10.4f}')
print(f'{"SymbioSLM":.<35} {"4.07M":>10} {"3.620":>10}')
print(f'{"JuliaSLM":.<35} {"5.04M":>10} {t_loss:>10.4f}')

# Upload to HuggingFace
HF_REPO = 'LisaMegaWatts/SymbioGPT-POC1-endosymbiosis'
os.makedirs('poc1_results', exist_ok=True)

# Save results metadata
results = {
    'method': 'organelle_specialist_fusion',
    'fused_config': {
        'd_model': FUSED_CONFIG.d_model, 'n_layers': FUSED_CONFIG.n_layers,
        'n_heads': FUSED_CONFIG.n_heads, 'head_dim': FUSED_CONFIG.head_dim,
        'organelles': list(FUSED_CONFIG.organelles), 'params': fused_params,
    },
    'specialists': {name: {'params': compute_symbio_params(SPECIALIST_CONFIGS[name]),
                           'val_loss': specialists[name][1]}
                    for name in specialists},
    'fused_val_loss': fused_loss,
    'teacher_val_loss': t_loss,
    'gate_entropy': gate_ent,
}
if has_baseline_a:
    results['baseline_scratch_val_loss'] = baseline_a_loss
if has_baseline_b:
    results['baseline_kd_val_loss'] = baseline_b_loss

with open('poc1_results/results.json', 'w') as f:
    json.dump(results, f, indent=2)

try:
    hf_api = HfApi()
    create_repo(HF_REPO, exist_ok=True)

    # Upload fused model
    if os.path.exists('fused-kd_best.pt'):
        hf_api.upload_file(
            path_or_fileobj='fused-kd_best.pt',
            path_in_repo='fused_best.pt',
            repo_id=HF_REPO,
            commit_message=f'POC1 fused model: val_loss={fused_loss:.4f}',
        )

    hf_api.upload_file(
        path_or_fileobj='poc1_results/results.json',
        path_in_repo='results.json',
        repo_id=HF_REPO,
        commit_message='POC1 results metadata',
    )
    print(f'\nUploaded to: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'HF upload failed: {e}')

print('\nDone!')